<a href="https://colab.research.google.com/github/cpython-projects/da_27_07_2026/blob/main/lesson_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Python DA #02: Робота з числами, похибками `float`, окргуленням та модулем `decimal` в Python

---

## 1. Об'єктна модель Python: `id`, `type` та `value`

В Python усе є об'єктом. Кожен об'єкт має три основні характеристики:

1. **Ідентифікатор (`id`)** — унікальна адреса об'єкта в пам'яті.
2. **Тип (`type`)** — визначає, які дані зберігає об'єкт та які операції з ним можливі.
3. **Значення (`value`)** — безпосередній вміст об'єкта.

In [3]:
x = 5

print(id(x))    # Повертає унікальний ID об'єкта в пам'яті (наприклад: 11645480)
print(type(x))  # Повертає тип даних: <class 'int'>

11645480
<class 'int'>


## 2. Особливості типу `float` та проблема точності

Числа з плаваючою крапкою (`float`) у комп'ютерах кодуються у двійковій системі числення за стандартом IEEE 754. Через це більшість десяткових дробі такий тип не може представити абсолютно точно.

### Проблема точного порівняння:

In [4]:
# Наївне порівняння через == дає False
print(0.1 + 0.1 + 0.1 == 0.3)  # False

# Якщо вивести фактичне значення суми:
print(0.1 + 0.1 + 0.1)  # 0.30000000000000004 (виникає накопичена похибка)


False
0.30000000000000004


### Як правильно порівнювати `float`?

Замість прямого порівняння `==` слід використовувати функцію `math.isclose()`, яка перевіряє, чи знаходяться значення достатньо близько одне до одного з урахуванням допустимої похибки:

In [ ]:
import math

print(math.isclose(0.1 + 0.1 + 0.1, 0.3))  # True

## 3. Оператори порівняння та логічний тип `bool`

Операції порівняння повертають булеві значення (`True` або `False`).

In [ ]:
# Оператори: >, >=, <, <=, ==, !=
print(3 > 6, 3 >= 6, 3 < 6, 3 <= 6, 3 == 6, 3 != 6)
# Результат: (False, False, True, True, False, True)

print(5 != 10)  # True

## 4. Округлення чисел: `math.ceil`, `math.floor` та `round`

При роботі з діленням нерідко виникає потреба округлити результат до цілого в той чи інший бік.

Приклад практики: підрахунок кількості коробок для товарів.

In [6]:
apples = 25
box = 6

print(apples / box)  # 4.166666666666667 (звичне ділення)

4.166666666666667


### Модуль `math`:

* **`math.ceil(x)`** (стеля) — округлює число **до найближчого більшого цілого** (вгору). Useful для логістики: якщо є 25 яблук і коробка вміщує 6, потрібно 5 коробок.
* **`math.floor(x)`** (підлога) — округлює число **до найближчого меншого цілого** (вниз).

In [7]:
import math

print(math.ceil(apples / box))   # 5
print(math.floor(apples / box))  # 4

5
4


### Вбудована функція `round()` та Банківське округлення (Banker's Rounding):

Вбудована функція `round()` у Python 3 використовує стратегію **"round half to even"** (округлення до найближчого парного числа). Це зменшує накопичувальну системну похибку при обробці великих масивів даних.

In [8]:
print(round(3.5))  # 4 (найближче парне ціле)
print(round(2.5))  # 2 (найближче парне ціле!)

4
2


## 5. Точні фінансові обчислення з модулем `decimal`

Коли ми працюємо з бізнес-задачами, фінансами чи комісіями, похибки `float` є неприпустимими. Для точних обчислень використовується модуль `decimal` та клас `Decimal`.

### Правильне створення `Decimal`:

Числа потрібно передавати **у вигляді рядків (`str`)**, а не чисел `float`, щоб уникнути похибки ще на етапі створення об'єкта.

In [9]:
from decimal import Decimal

# Неправильно: Decimal(0.1) — вже містить похибку float
# Правильно:
x = Decimal('0.1')
print(x + x + x)  # 0.3 (точно!)

0.3


## 6. Стратегії округлення у `Decimal` (`quantize`)

Метод `.quantize()` дозволяє округляти числа `Decimal` до необхідної точності (наприклад, до 2 знаків після коми) з явно вказаним режимом округлення.

### Порівняльні режими округлення:

1. **`ROUND_HALF_UP`** — класичне шкільне округлення (0.5 і вище йде вгору).
2. **`ROUND_UP`** — завжди округлює від нуля (по модулю в більшу сторону).
3. **`ROUND_CEILING`** — завжди округлює в сторону $+\infty$ (на математичній осі).

### Приклад з фінансовою комісією на обсязі в 1 000 000 транзакцій:

In [10]:
from decimal import Decimal, ROUND_HALF_UP, ROUND_UP, ROUND_CEILING

operations = 1_000_000
fee = Decimal('0.1234')  # Базова комісія

# 1. Стандартне округлення (ROUND_HALF_UP)
standard_fee = fee.quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)
print(standard_fee, standard_fee * operations)
# Результат: 0.12 грн/транзакція -> 120 000.00 грн сумарно

# 2. Округлення завжди вгору (ROUND_UP)
up_fee = fee.quantize(Decimal('0.01'), rounding=ROUND_UP)
print(up_fee, up_fee * operations)
# Результат: 0.13 грн/транзакція -> 130 000.00 грн сумарно (прибуток компанії вище на 10 000 грн!)


0.12 120000.00
0.13 130000.00


### Різниця між `ROUND_UP` та `ROUND_CEILING` (важливо для від'ємних чисел):

* **`ROUND_UP`**: Округлює від нуля за модулем.
* **`ROUND_CEILING`**: Округлює в напрямку $+\infty$.

In [11]:
# Додатні числа (поведінка однакова):
x_pos = Decimal('1.21')
print(x_pos.quantize(Decimal('0.1'), rounding=ROUND_UP))       # 1.3
print(x_pos.quantize(Decimal('0.1'), rounding=ROUND_CEILING))  # 1.3

# Від'ємні числа (поведінка ВІДРІЗНЯЄТЬСЯ):
x_neg = Decimal('-1.21')
print(x_neg.quantize(Decimal('0.1'), rounding=ROUND_UP))       # -1.3 (від нуля далі)
print(x_neg.quantize(Decimal('0.1'), rounding=ROUND_CEILING))  # -1.2 (у бік +нескінченності, тобто більше число)

1.3
1.3
-1.3
-1.2


## 7. Допоміжні та математичні функції

Вбудовані функції Python для базових арифметичних задач:

In [2]:
# 1. Піднесення до степеня / Добування кореня
print(7 ** 0.5)      # 2.6457513110645907 (квадратний корінь)
print(7 ** (1/3))    # 1.912931182772389  (кубічний корінь)

# 2. Пошук мінімуму та максимуму
print(min(23, 1, 8, -4))  # -4
print(max(23, 1, 8, -4))  # 23

# 3. Абсолютне значення (модуль числа)
print(abs(-5))  # 5
print(abs(5))   # 5

2.6457513110645907
1.912931182772389
-4
23
5
5
